In [27]:
# 🔹 1. Import Libraries
import pandas as pd
import numpy as np
import os
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 🔹 2. Check files in directory
print("Files in folder:", os.listdir())

# 🔹 3. Load Dataset
# Added engine='python' and on_bad_lines='skip' to handle potential parsing errors in the CSV files.
true_df = pd.read_csv("True.csv", engine='python', on_bad_lines='skip')
fake_df = pd.read_csv("Fake.csv", engine='python', on_bad_lines='skip')

# 🔹 4. Add Labels
true_df["label"] = 1   # Real News
fake_df["label"] = 0   # Fake News

# 🔹 5. Combine Datasets
df = pd.concat([true_df, fake_df], axis=0)

# 🔹 6. Shuffle Data
df = df.sample(frac=1).reset_index(drop=True)

# 🔹 7. Combine Title + Text (Better Accuracy)
df["content"] = df["title"] + " " + df["text"]

# 🔹 8. Text Cleaning Function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    return text

# Apply cleaning
df["content"] = df["content"].apply(clean_text)

# 🔹 9. Features & Labels
X = df["content"]
y = df["label"]

# 🔹 10. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 🔹 11. Convert Text → Numbers (TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 🔹 12. Train Model
model = LogisticRegression()
model.fit(X_train_vec, y_train)

# 🔹 13. Predictions
y_pred = model.predict(X_test_vec)

# 🔹 14. Evaluate
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 🔹 15. Predict Custom News
def predict_news(news):
    news = clean_text(news)
    news_vec = vectorizer.transform([news])
    prediction = model.predict(news_vec)

    if prediction[0] == 1:
        print("\u2705 Real News")
    else:
        print("\u274c Fake News")

# 🔹 Example Test
print("\n--- Testing Custom Input ---")
predict_news("Breaking: Government passes new economic reform bill")

# 🔹 16. Save Model
import pickle

pickle.dump(model, open("fake_news_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

print("\nModel saved successfully!")

Files in folder: ['.config', 'True.csv', 'Fake.csv', 'sample_data']

Accuracy: 0.9921135646687698

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.99      0.99      1218
           1       0.99      1.00      0.99      1318

    accuracy                           0.99      2536
   macro avg       0.99      0.99      0.99      2536
weighted avg       0.99      0.99      0.99      2536


--- Testing Custom Input ---
✅ Real News

Model saved successfully!
